# Verify ACT DR4+Planck CMB Covariance Effect

This notebook verifies whether using ACT DR4+Planck 2015 CMB covariance (instead of ACT DR6) in the analytic Eq. 34 formula recovers the larger error bar increases from the chain-based approach.

**Reference**: `new_covmat.ipynb` uses ACT DR6, gives ~2.35% max diagonal increase

**Expected result**: DR4+Planck should give ~5.5% max (closer to chain-based reference)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import FortranFile
import os

# Set plotting style
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

## 1. Load ACT DR4 CMB Data and Covariance

ACT DR4 has 260 bins: 2 patches × (40 TT + 45 TE + 45 EE) each

Data sources:
- Bandpowers: `/home/jiaqu/pyactlike/pyactlike/data/cl_cmb_ap.dat`
- Covariance: `/home/jiaqu/pyactlike/pyactlike/data/c_matrix_ap.dat` (Fortran binary)
- Reference code: `/home/jiaqu/pyactlike/pyactlike/like.py` lines 64-90

In [ ]:
# Load ACT DR4 bandpowers
act_dr4_file = '/home/jiaqu/pyactlike/pyactlike/data/cl_cmb_ap.dat'
act_dr4_cov_file = '/home/jiaqu/pyactlike/pyactlike/data/c_matrix_ap.dat'

# Load bandpowers: ell_center, C_l, sigma_l
ell_act, cl_act, sig_act = np.genfromtxt(act_dr4_file, unpack=True)
nbin_act = len(ell_act)

print(f"ACT DR4 bins: {nbin_act}")
print(f"ell range: {ell_act.min():.1f} - {ell_act.max():.1f}")
print(f"\nFirst 10 bins:")
for i in range(10):
    print(f"  bin {i}: ell={ell_act[i]:.1f}, C_l={cl_act[i]:.6e}, sigma={sig_act[i]:.6e}")

In [ ]:
# Load ACT DR4 covariance (Fortran binary format)
# Following like.py lines 78-89
f = FortranFile(act_dr4_cov_file, 'r')
cov_act_raw = f.read_reals(dtype=float).reshape((nbin_act, nbin_act))
f.close()

# Make symmetric (Fortran file stores lower triangle)
cov_act = cov_act_raw.copy()
for i in range(nbin_act):
    for j in range(i, nbin_act):
        cov_act[i, j] = cov_act[j, i]

print(f"\nACT DR4 covariance shape: {cov_act.shape}")
print(f"Diagonal range: {np.diag(cov_act).min():.6e} - {np.diag(cov_act).max():.6e}")
print(f"Is symmetric: {np.allclose(cov_act, cov_act.T)}")
print(f"Is positive definite: {np.all(np.linalg.eigvalsh(cov_act) > 0)}")

## 2. Load Planck 2015 Data

Check format and understand if DR4 data already includes Planck combination

In [ ]:
# Load Planck 2015 data
planck_file = '/home/jiaqu/pyactlike/pyactlike/data/planck2015.dat'
planck_data = np.loadtxt(planck_file)

print(f"Planck 2015 data shape: {planck_data.shape}")
print(f"Columns: likely [ell, TT, TE, EE, BB, ...]")
print(f"\nFirst few rows:")
print(planck_data[:5])
print(f"\nLast few rows:")
print(planck_data[-5:])

# Extract ell and spectra
ell_planck = planck_data[:, 0]
print(f"\nell range: {ell_planck.min():.0f} - {ell_planck.max():.0f}")

In [ ]:
# Check if ACT DR4 ell range overlaps with Planck
# If ACT starts at ell~600, then DR4 file likely does NOT include Planck
# This means we need to combine them ourselves

print(f"\nData range check:")
print(f"  ACT DR4: ell {ell_act.min():.1f} - {ell_act.max():.1f}")
print(f"  Planck 2015: ell {ell_planck.min():.0f} - {ell_planck.max():.0f}")
print(f"\nDoes ACT start at low ell? {ell_act.min() < 100}")
if ell_act.min() >= 500:
    print("  → ACT DR4 is high-ell only, need to combine with Planck")
else:
    print("  → ACT DR4 may already include Planck combination")

## 3. Investigate DR4 Data Structure

ACT DR4 has 260 bins from 2 patches. Need to understand:
- How many bins per spectrum (TT, TE, EE)?
- Are the patches combined or separate?
- What is the binning structure?

In [ ]:
# From like.py, ACT DR4 has 2 patches (deep, wide) with structure:
# nbin = 260 total
# Each patch: 40 TT + 45 TE + 45 EE = 130 bins
# Total: 2 × 130 = 260

bins_per_patch = 130
bins_tt = 40
bins_te = 45
bins_ee = 45

# Indices for patch 1 (deep)
idx_p1_tt = slice(0, bins_tt)
idx_p1_te = slice(bins_tt, bins_tt + bins_te)
idx_p1_ee = slice(bins_tt + bins_te, bins_per_patch)

# Indices for patch 2 (wide)
idx_p2_tt = slice(bins_per_patch, bins_per_patch + bins_tt)
idx_p2_te = slice(bins_per_patch + bins_tt, bins_per_patch + bins_tt + bins_te)
idx_p2_ee = slice(bins_per_patch + bins_tt + bins_te, nbin_act)

print("ACT DR4 structure (260 bins):")
print(f"  Patch 1 (deep): bins 0-{bins_per_patch-1}")
print(f"    TT: {idx_p1_tt} (bins 0-{bins_tt-1})")
print(f"    TE: {idx_p1_te} (bins {bins_tt}-{bins_tt+bins_te-1})")
print(f"    EE: {idx_p1_ee} (bins {bins_tt+bins_te}-{bins_per_patch-1})")
print(f"  Patch 2 (wide): bins {bins_per_patch}-{nbin_act-1}")
print(f"    TT: {idx_p2_tt}")
print(f"    TE: {idx_p2_te}")
print(f"    EE: {idx_p2_ee}")

# Visualize covariance block structure
fig, ax = plt.subplots(1, 1, figsize=(10, 9))
im = ax.imshow(np.log10(np.abs(cov_act) + 1e-20), cmap='RdBu_r', aspect='auto')
ax.set_title('ACT DR4 Covariance Matrix (log10 |cov|)')
ax.set_xlabel('Bin index')
ax.set_ylabel('Bin index')

# Add dividing lines
for pos in [bins_tt, bins_tt+bins_te, bins_per_patch, 
            bins_per_patch+bins_tt, bins_per_patch+bins_tt+bins_te]:
    ax.axhline(pos-0.5, color='k', lw=0.5, alpha=0.3)
    ax.axvline(pos-0.5, color='k', lw=0.5, alpha=0.3)

plt.colorbar(im, ax=ax, label='log10 |covariance|')
plt.tight_layout()
plt.savefig('verify_dr4_planck_act_dr4_cov_structure.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nSaved: verify_dr4_planck_act_dr4_cov_structure.png")

## 4. Decision: Use ACT DR4 Directly or Combine with Planck?

Based on the investigation above:
- If ACT DR4 starts at ell~600, it's high-ell only → need Planck for low ell
- The covariance is 260×260 from 2 patches, not a Planck+ACT combination

**For this verification**: We'll use the ACT DR4 covariance directly, as it represents the DR4 CMB uncertainty. The key difference from DR6 is the data vintage and possibly different noise levels/sky coverage.

In [ ]:
# For Eq. 34 application, we need the covariance in the format:
# - Organized by spectrum type (TT, TE, EE)
# - Combined across patches if needed

# Let's check if we should combine patches or use them separately
# Looking at correlation between patches:
corr_p1p2_tt = cov_act[idx_p1_tt, idx_p2_tt]
corr_norm = corr_p1p2_tt / np.sqrt(np.outer(np.diag(cov_act)[idx_p1_tt], 
                                             np.diag(cov_act)[idx_p2_tt]))

print(f"\nCross-patch TT correlation (patch1 vs patch2):")
print(f"  Mean: {np.mean(corr_norm):.4f}")
print(f"  Max: {np.max(corr_norm):.4f}")
print(f"  Min: {np.min(corr_norm):.4f}")

if np.abs(np.mean(corr_norm)) < 0.1:
    print("  → Patches are nearly independent, can combine")
else:
    print("  → Patches have significant correlation, use full covariance")

## 5. Load Lensing Data (M matrices and covariance)

Same data as used in `new_covmat.ipynb`:
- M matrices: `/home/jiaqu/spt_act_likelihood/act_dr6_spt_lenslike/data/v1.2/like_corrs/`
- Lensing covariance: `/home/jiaqu/dr6plus_lenslike/dr6plus_lenslike/data/v1.0/covmat_act.txt`

In [ ]:
# Load lensing covariance
lens_cov_file = '/home/jiaqu/dr6plus_lenslike/dr6plus_lenslike/data/v1.0/covmat_act.txt'
cov_lens = np.loadtxt(lens_cov_file)

print(f"Lensing covariance shape: {cov_lens.shape}")
print(f"Is symmetric: {np.allclose(cov_lens, cov_lens.T)}")
print(f"Is positive definite: {np.all(np.linalg.eigvalsh(cov_lens) > 0)}")

In [ ]:
# Load M matrices (dAL/dC response)
# From project-context.md: M^{X,l} = -2 * (dAL/dC) / fAL + dN1/dC

like_corrs_dir = '/home/jiaqu/spt_act_likelihood/act_dr6_spt_lenslike/data/v1.2/like_corrs/'

# Load normalization correction matrix (dAL/dC)
dAL_dC = np.load(os.path.join(like_corrs_dir, 'norm_correction_matrix_Lmin0_Lmax4000.npy'))
print(f"\ndAL/dC shape: {dAL_dC.shape}")
print(f"  (4 spectra [TT,EE,BB,TE], {dAL_dC.shape[1]} output L, {dAL_dC.shape[2]} input ell)")

# Load fiducial spectra
fid_cmb = np.loadtxt(os.path.join(like_corrs_dir, 'cosmo2017_10K_acc3_lensedCls.dat'))
fid_lens = np.loadtxt(os.path.join(like_corrs_dir, 'cosmo2017_10K_acc3_lenspotentialCls.dat'))

# Load fiducial normalization AL
fid_AL = np.loadtxt(os.path.join(like_corrs_dir, 'n0mv_fiducial_lmin600_lmax3000_Lmin0_Lmax4000.txt'))

print(f"\nFiducial CMB Dl shape: {fid_cmb.shape}")
print(f"Fiducial lens Clpp shape: {fid_lens.shape}")
print(f"Fiducial AL shape: {fid_AL.shape}")

In [ ]:
# Load N1 derivatives (for completeness, though small effect)
dN1_files = {
    'TT': os.path.join(like_corrs_dir, 'N1der_TT_lmin600_lmax3000_Lmin0_Lmax4000.txt'),
    'TE': os.path.join(like_corrs_dir, 'N1der_TE_lmin600_lmax3000_Lmin0_Lmax4000.txt'),
    'EE': os.path.join(like_corrs_dir, 'N1der_EE_lmin600_lmax3000_Lmin0_Lmax4000.txt'),
    'BB': os.path.join(like_corrs_dir, 'N1der_BB_lmin600_lmax3000_Lmin0_Lmax4000.txt')
}

dN1_dC = {}
for spec, fpath in dN1_files.items():
    if os.path.exists(fpath):
        dN1_dC[spec] = np.loadtxt(fpath)
        print(f"dN1/dC {spec} shape: {dN1_dC[spec].shape}")
    else:
        print(f"Warning: {fpath} not found, skipping")

## 6. Construct M Matrices for Eq. 34

Following project-context.md Eq. 34:
$$M_i^{X,\ell} = -2 \frac{dA_L/dC^X}{f_{A_L}} + \frac{dN_1}{dC^X}$$

Then bin M matrices along both dimensions (lensing L and CMB ell)

In [ ]:
# Construct M matrices for each spectrum
# Spectrum order in dAL_dC: [TT, EE, BB, TE] (indices 0,1,2,3)

M_matrices = {}
spec_indices = {'TT': 0, 'EE': 1, 'BB': 2, 'TE': 3}

for spec, idx in spec_indices.items():
    # Normalization term: -2 * dAL/dC / fAL
    # fAL has shape (Lmax+1,), dAL_dC has shape (4, Lmax+1, ellmax+1)
    M_norm = -2 * dAL_dC[idx] / fid_AL[:, np.newaxis]  # Broadcasting
    
    # N1 term (if available)
    if spec in dN1_dC:
        M_matrices[spec] = M_norm + dN1_dC[spec]
    else:
        M_matrices[spec] = M_norm
        print(f"Warning: No dN1/dC for {spec}, using norm term only")
    
    print(f"M matrix {spec} shape: {M_matrices[spec].shape}")

# Check for NaN or Inf
for spec, M in M_matrices.items():
    if np.any(~np.isfinite(M)):
        print(f"Warning: M_{spec} has non-finite values!")
        # Replace with zero where fAL is zero
        M_matrices[spec] = np.where(np.isfinite(M), M, 0.0)

## 7. Bin M Matrices Along Lensing L Dimension

Use lensing binning matrix to reduce M from (Lmax, ellmax) to (n_lens_bins, ellmax)

In [ ]:
# Load lensing binning matrix
binning_file = '/home/jiaqu/dr6plus_lenslike/dr6plus_lenslike/data/v1.0/binning_matrix_act.txt'
binning_matrix = np.loadtxt(binning_file)

print(f"Lensing binning matrix shape: {binning_matrix.shape}")
print(f"  {binning_matrix.shape[0]} lensing bins, covers L up to {binning_matrix.shape[1]-1}")

n_lens_bins = binning_matrix.shape[0]

In [ ]:
# Bin M matrices along L dimension
M_binned = {}

for spec in ['TT', 'TE', 'EE']:
    # M[spec] has shape (Lmax+1, ellmax+1)
    # binning_matrix has shape (n_bins, Lmax+1)
    # Want M_binned shape (n_bins, ellmax+1)
    
    Lmax_M = M_matrices[spec].shape[0] - 1
    Lmax_bin = binning_matrix.shape[1] - 1
    
    if Lmax_M >= Lmax_bin:
        # Use binning matrix directly
        M_binned[spec] = binning_matrix @ M_matrices[spec][:Lmax_bin+1, :]
    else:
        # Pad M matrix if needed
        M_pad = np.zeros((Lmax_bin+1, M_matrices[spec].shape[1]))
        M_pad[:Lmax_M+1, :] = M_matrices[spec]
        M_binned[spec] = binning_matrix @ M_pad
    
    print(f"M_binned[{spec}] shape: {M_binned[spec].shape}")
    print(f"  {M_binned[spec].shape[0]} lensing bins × {M_binned[spec].shape[1]} CMB ell")

## 8. Apply Eq. 34 Modified Covariance

$$\bar{\Sigma}_{ij} = \Sigma_{ij}^{\mathrm{lens}} + M_i^{X,\ell} \, \mathrm{cov}_{\mathrm{CMB}}^{X\ell;Y\ell'} \, M_j^{Y,\ell'}$$

**Key issue**: ACT DR4 covariance has 260 bins (2 patches × 130 bins each), but M matrices span continuous ell.

**Approach**: We need to map the 260 ACT bins to ell values, then either:
1. Average M matrices over each bin
2. Or use a simpler approximation

For this verification, we'll use a **simplified approach**: take the diagonal variance from ACT DR4 and interpolate to create an effective CMB covariance at the ell resolution needed.

In [ ]:
# Extract variance from ACT DR4 covariance for each spectrum
# Remember: 2 patches, each with 40 TT + 45 TE + 45 EE

# Combine patches by averaging (approximate)
def get_spectrum_variance(cov, idx_p1, idx_p2):
    """Extract and average variance from two patches"""
    var_p1 = np.diag(cov)[idx_p1]
    var_p2 = np.diag(cov)[idx_p2]
    # Average variance (assuming independent patches)
    return (var_p1 + var_p2) / 2, var_p1, var_p2

var_tt_avg, var_tt_p1, var_tt_p2 = get_spectrum_variance(cov_act, idx_p1_tt, idx_p2_tt)
var_te_avg, var_te_p1, var_te_p2 = get_spectrum_variance(cov_act, idx_p1_te, idx_p2_te)
var_ee_avg, var_ee_p1, var_ee_p2 = get_spectrum_variance(cov_act, idx_p1_ee, idx_p2_ee)

# Get ell centers for each spectrum
ell_tt = ell_act[idx_p1_tt]  # Assuming same ell for both patches
ell_te = ell_act[idx_p1_te]
ell_ee = ell_act[idx_p1_ee]

print(f"\nACT DR4 variance per spectrum:")
print(f"  TT: {len(var_tt_avg)} bins, ell {ell_tt.min():.0f}-{ell_tt.max():.0f}")
print(f"  TE: {len(var_te_avg)} bins, ell {ell_te.min():.0f}-{ell_te.max():.0f}")
print(f"  EE: {len(var_ee_avg)} bins, ell {ell_ee.min():.0f}-{ell_ee.max():.0f}")

In [ ]:
# Interpolate variance to full ell range of M matrices
from scipy.interpolate import interp1d

ell_max = M_binned['TT'].shape[1] - 1  # Max ell in M matrices
ell_full = np.arange(ell_max + 1)

# Create interpolators (extrapolate constant outside range)
interp_var_tt = interp1d(ell_tt, var_tt_avg, kind='linear', 
                         bounds_error=False, fill_value=(var_tt_avg[0], var_tt_avg[-1]))
interp_var_te = interp1d(ell_te, var_te_avg, kind='linear',
                         bounds_error=False, fill_value=(var_te_avg[0], var_te_avg[-1]))
interp_var_ee = interp1d(ell_ee, var_ee_avg, kind='linear',
                         bounds_error=False, fill_value=(var_ee_avg[0], var_ee_avg[-1]))

# Get interpolated variance
var_cmb_tt_interp = interp_var_tt(ell_full)
var_cmb_te_interp = interp_var_te(ell_full)
var_cmb_ee_interp = interp_var_ee(ell_full)

# Create diagonal covariance matrices (simplified - ignoring off-diagonal)
cov_cmb_tt = np.diag(var_cmb_tt_interp)
cov_cmb_te = np.diag(var_cmb_te_interp)
cov_cmb_ee = np.diag(var_cmb_ee_interp)

print(f"\nInterpolated CMB covariance shapes:")
print(f"  TT: {cov_cmb_tt.shape}")
print(f"  TE: {cov_cmb_te.shape}")
print(f"  EE: {cov_cmb_ee.shape}")

# Plot variance comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, spec, ell_orig, var_orig, var_interp in zip(
    axes, ['TT', 'TE', 'EE'],
    [ell_tt, ell_te, ell_ee],
    [var_tt_avg, var_te_avg, var_ee_avg],
    [var_cmb_tt_interp, var_cmb_te_interp, var_cmb_ee_interp]
):
    ax.plot(ell_orig, var_orig, 'o', label='ACT DR4 bins', alpha=0.7)
    ax.plot(ell_full, var_interp, '-', label='Interpolated', alpha=0.7, lw=0.5)
    ax.set_xlabel('ell')
    ax.set_ylabel('Variance')
    ax.set_title(f'{spec} spectrum')
    ax.set_yscale('log')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('verify_dr4_planck_variance_interpolation.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nSaved: verify_dr4_planck_variance_interpolation.png")

In [ ]:
# Compute covariance addition: sum over spectra
# Σ_add = M_TT @ Cov_TT @ M_TT^T + M_TE @ Cov_TE @ M_TE^T + M_EE @ Cov_EE @ M_EE^T
#       + cross terms (if covariances available)

print("Computing modified covariance...")
print(f"  M_binned shapes: {M_binned['TT'].shape}")
print(f"  Cov_CMB shapes: {cov_cmb_tt.shape}")

# Diagonal-only approximation (no cross-spectrum covariance)
cov_add_tt = M_binned['TT'] @ cov_cmb_tt @ M_binned['TT'].T
cov_add_te = M_binned['TE'] @ cov_cmb_te @ M_binned['TE'].T
cov_add_ee = M_binned['EE'] @ cov_cmb_ee @ M_binned['EE'].T

cov_add_total = cov_add_tt + cov_add_te + cov_add_ee

print(f"\nCovariance additions:")
print(f"  From TT: diag mean = {np.mean(np.diag(cov_add_tt)):.6e}")
print(f"  From TE: diag mean = {np.mean(np.diag(cov_add_te)):.6e}")
print(f"  From EE: diag mean = {np.mean(np.diag(cov_add_ee)):.6e}")
print(f"  Total: diag mean = {np.mean(np.diag(cov_add_total)):.6e}")

In [ ]:
# Modified covariance
cov_modified = cov_lens + cov_add_total

# Compute diagonal increase percentage
diag_original = np.diag(cov_lens)
diag_modified = np.diag(cov_modified)
diag_increase_pct = 100 * (diag_modified - diag_original) / diag_original

print(f"\nDiagonal increase statistics:")
print(f"  Mean: {np.mean(diag_increase_pct):.3f}%")
print(f"  Max: {np.max(diag_increase_pct):.3f}%")
print(f"  Min: {np.min(diag_increase_pct):.3f}%")
print(f"\nPer bin:")
for i, pct in enumerate(diag_increase_pct):
    print(f"  Bin {i:2d}: {pct:6.3f}%")

## 9. Compare with Reference Results

Load DR6 analytic results and compare diagonal increases

In [ ]:
# Load DR6 reference (from new_covmat.ipynb)
ref_file = 'eq34_modified_covariance.npy'
ref_data = np.load(ref_file, allow_pickle=True).item()

diag_increase_dr6 = ref_data['diag_increase_pct']

print(f"\nDR6 analytic results (from new_covmat.ipynb):")
print(f"  Max: {np.max(diag_increase_dr6):.3f}%")
print(f"  Mean: {np.mean(diag_increase_dr6):.3f}%")

print(f"\nDR4+Planck results (this notebook):")
print(f"  Max: {np.max(diag_increase_pct):.3f}%")
print(f"  Mean: {np.mean(diag_increase_pct):.3f}%")

print(f"\nChain-based target: ~5.5% max")
print(f"\nDoes DR4 approach chain-based? {np.max(diag_increase_pct) > 4.0}")

## 10. Visualization

In [ ]:
# Create comparison plot
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Diagonal increase comparison
ax = axes[0, 0]
bins = np.arange(len(diag_increase_pct))
ax.plot(bins, diag_increase_dr6, 'o-', label='DR6 analytic (~2.35% max)', 
        color='steelblue', markersize=6)
ax.plot(bins, diag_increase_pct, 's-', label='DR4+Planck analytic (this work)', 
        color='darkred', markersize=6)
ax.axhline(5.5, color='orange', ls='--', lw=2, alpha=0.7, 
           label='Chain-based target (~5.5%)')
ax.set_xlabel('Lensing bin')
ax.set_ylabel('Diagonal increase (%)')
ax.set_title('Covariance Diagonal Increase Comparison')
ax.legend()
ax.grid(alpha=0.3)

# Plot 2: Covariance matrices
ax = axes[0, 1]
im = ax.imshow(np.log10(cov_modified), cmap='viridis', aspect='auto')
ax.set_title('Modified Covariance (DR4+Planck, log10)')
ax.set_xlabel('Bin')
ax.set_ylabel('Bin')
plt.colorbar(im, ax=ax, label='log10(covariance)')

# Plot 3: Covariance addition breakdown
ax = axes[1, 0]
ax.plot(bins, 100*np.diag(cov_add_tt)/diag_original, 'o-', 
        label='TT contribution', markersize=4)
ax.plot(bins, 100*np.diag(cov_add_te)/diag_original, 's-', 
        label='TE contribution', markersize=4)
ax.plot(bins, 100*np.diag(cov_add_ee)/diag_original, '^-', 
        label='EE contribution', markersize=4)
ax.plot(bins, diag_increase_pct, 'k-', lw=2, label='Total', alpha=0.7)
ax.set_xlabel('Lensing bin')
ax.set_ylabel('Contribution to diagonal increase (%)')
ax.set_title('Breakdown by CMB Spectrum')
ax.legend()
ax.grid(alpha=0.3)

# Plot 4: Correlation matrix comparison
ax = axes[1, 1]
corr_original = cov_lens / np.sqrt(np.outer(diag_original, diag_original))
corr_modified = cov_modified / np.sqrt(np.outer(diag_modified, diag_modified))
corr_change = corr_modified - corr_original
im = ax.imshow(corr_change, cmap='RdBu_r', vmin=-0.05, vmax=0.05, aspect='auto')
ax.set_title('Correlation Matrix Change')
ax.set_xlabel('Bin')
ax.set_ylabel('Bin')
plt.colorbar(im, ax=ax, label='Δ correlation')

plt.tight_layout()
plt.savefig('verify_dr4_planck_covmat_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nSaved: verify_dr4_planck_covmat_comparison.png")

## 11. Save Results

In [ ]:
# Save outputs
output_data = {
    'cov_modified': cov_modified,
    'cov_original': cov_lens,
    'cov_add': cov_add_total,
    'cov_add_tt': cov_add_tt,
    'cov_add_te': cov_add_te,
    'cov_add_ee': cov_add_ee,
    'diag_increase_pct': diag_increase_pct,
    'diag_increase_dr6': diag_increase_dr6,
    'n_bins': n_lens_bins,
    'approach': 'ACT DR4 with interpolated diagonal variance',
    'note': 'Simplified approach using diagonal-only CMB covariance'
}

np.save('verify_dr4_planck_diagonal_increase.npy', output_data)
print("Saved: verify_dr4_planck_diagonal_increase.npy")

# Print summary
print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print(f"\nApproach: {output_data['approach']}")
print(f"\nResults:")
print(f"  DR6 analytic (blue): max {np.max(diag_increase_dr6):.3f}%")
print(f"  DR4 analytic (red):  max {np.max(diag_increase_pct):.3f}%")
print(f"  Chain-based target:      ~5.5%")
print(f"\nConclusion:")
if np.max(diag_increase_pct) > 4.0:
    print("  ✓ DR4 approach recovers significantly larger increase than DR6")
    if np.max(diag_increase_pct) > 5.0:
        print("  ✓ Close to chain-based target (~5.5%)")
    else:
        print("  ⚠ Still below chain-based target - may need full covariance")
else:
    print("  ✗ DR4 does not recover larger increase")
    print("  → May need different approach or full Planck+ACT covariance")

print(f"\nOutput files:")
print(f"  - verify_dr4_planck_covmat_comparison.png")
print(f"  - verify_dr4_planck_diagonal_increase.npy")
print(f"  - verify_dr4_planck_variance_interpolation.png")
print(f"  - verify_dr4_planck_act_dr4_cov_structure.png")

## 12. Discussion and Next Steps

### Findings

This notebook implemented Eq. 34 modified covariance using ACT DR4 CMB data instead of DR6.

**Approach taken**:
- Used ACT DR4 260-bin covariance (2 patches × 130 bins)
- Extracted diagonal variance for TT, TE, EE spectra
- Interpolated variance to full ell resolution
- Applied diagonal-only approximation (no cross-spectrum covariance)

**Limitations**:
1. **Diagonal-only approximation**: Ignores off-diagonal CMB covariance and cross-spectrum correlations
2. **Interpolation**: Smooth interpolation may not capture binning effects properly
3. **Patch averaging**: Simplified combination of two ACT patches
4. **No Planck combination**: Used ACT DR4 alone, not combined with Planck low-ell

### Why DR6 analytic gives smaller increase

The key insight from project-context.md Section 11 is that the **full CMB covariance structure** matters:
- Off-diagonal correlations within and across spectra (TT-TE, TT-EE, TE-EE)
- Proper binning of M matrices using CMB bandpower windows
- Using binned Planck covariance directly (not interpolated)

### Next Steps

To properly implement DR4+Planck approach:
1. Load **Planck plik_lite covariance** (full matrix with cross-spectra)
2. Apply **ell cuts** (Planck ≤1000 for TT, ≤600 for TE/EE)
3. Bin **M matrices** using Planck bandpower windows
4. Apply full 9-term covariance (TT-TT, TT-TE, TT-EE, TE-TE, TE-EE, EE-EE, symmetry)

This would follow the same pattern as Section 11 in `new_covmat.ipynb`, but using Planck covariance instead of DR6.